# Baseline model

We will use a TF+IDF + Logistic Regression model as a baseline.

## Load and split the preprocessed data (no tokenization)

In [ ]:
import pandas as pd

data = pd.read_csv("../data/preproc_train.csv")
data.head()

In [ ]:
text = data["comment_text"]

labels = data[
    ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
]

In [ ]:
text = text.fillna("")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    text,
    labels,
    test_size=0.1,
    random_state=42
)

In [ ]:
y_train.shape

## Use TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    max_features=30_000,
    analyzer='word',
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.9,
    token_pattern=r'\w{1,}'             # ignore overly common words
)


In [ ]:
X_train_vectorized = vectorizer.fit_transform(X_train)
X_valid_vectorized = vectorizer.transform(X_valid)

print(X_train.shape, X_valid.shape)

## Train a Logistic Regression model

Because we have 6 labels, use One-vs-Rest Logistic Regression.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=500,
        class_weight="balanced"
    )
)

model.fit(X_train_vectorized, y_train)

In [ ]:
y_probs = model.predict_proba(X_valid_vectorized)
y_probs.shape

In [ ]:
y_probs[0]

## Evaluating useing macro PR-AUC metric

In [ ]:
from sklearn.metrics import classification_report
y_pred = (y_probs > 0.5).astype(int)  # Threshold probabilities for predictions
print(classification_report(y_valid, y_pred, target_names=labels.columns))

In [ ]:
from sklearn.metrics import average_precision_score


pr_auc_scores = {}
for i, label in enumerate(labels.columns):
    score = average_precision_score(y_valid[label], y_probs[:, i])
    pr_auc_scores[label] = score
    print(f"PR-AUC for {label}: {score:.4f}")

macro_pr_auc = sum(pr_auc_scores.values()) / len(pr_auc_scores)
print(f"Macro PR-AUC: {macro_pr_auc:.4f}")